# Customer Lifetime Value & Next-Best-Action Engine

**Predict near-term customer value and turn it into an actionable retention / growth queue.**

### Commercial questions
1. Which customers are likely to buy again in the next 90 days?
2. How much revenue is each customer expected to generate?
3. Which customers combine high value with elevated lapse risk?
4. What is the most sensible next action: retain, cross-sell, win back, basket-build, or low-cost nurture?
5. Can customer behaviour outperform simple historical-spend ranking?

### Real dataset
This notebook uses **UCI Online Retail II**, containing **1,067,371 transactions** from a UK-based non-store retailer between December 2009 and December 2011.

### Privacy / modelling policy
- **No age, DOB, gender or personal demographic predictors**
- raw Customer IDs are grouping keys only, never model inputs
- final action tables use anonymised customer keys
- all predictive features describe purchase behaviour before each cutoff

### System design
- rolling customer snapshots
- RFM + basket + cadence + product-diversity features
- 90-day repeat-purchase propensity
- 90-day customer-value prediction
- Logistic Regression and XGBoost baselines/challengers
- ROC-AUC, PR-AUC, MAE, WMAPE and decile-lift evaluation
- SHAP explanations
- next-best-action policy and prioritised CRM queue

In [ ]:
# Colab / GitHub Actions safe setup
import sys, subprocess, importlib.util

required = {
    "openpyxl": "openpyxl",
    "xgboost": "xgboost",
    "shap": "shap"
}
for module, package in required.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("Environment ready.")

In [ ]:
import warnings, zipfile, urllib.request, hashlib
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_score, recall_score,
    f1_score, mean_absolute_error, mean_squared_error
)
from xgboost import XGBClassifier, XGBRegressor

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

print("Libraries imported.")

## 1. Download UCI Online Retail II

The source file is downloaded directly from the UCI Machine Learning Repository. It contains two Excel sheets covering two years of transactions.

In [ ]:
DATA_URL = "https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip"
zip_path = Path("online_retail_ii.zip")
extract_dir = Path("online_retail_ii_data")

if not zip_path.exists():
    urllib.request.urlretrieve(DATA_URL, zip_path)

extract_dir.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(extract_dir)

xlsx_files = list(extract_dir.rglob("*.xlsx"))
if not xlsx_files:
    raise FileNotFoundError("Online Retail II XLSX was not found.")

xlsx_path = xlsx_files[0]
book = pd.ExcelFile(xlsx_path, engine="openpyxl")
print("Sheets:", book.sheet_names)

parts = [pd.read_excel(xlsx_path, sheet_name=s, engine="openpyxl") for s in book.sheet_names]
raw = pd.concat(parts, ignore_index=True)

print(f"Raw rows: {len(raw):,}")
print("Columns:", raw.columns.tolist())
display(raw.head())

## 2. Clean transactions and remove returns / cancellations

Only completed positive-value purchases with a customer identifier are used. The modelling population is restricted to UK customers to keep currency / market behaviour consistent.

In [ ]:
df = raw.copy()

# Normalise column names across the two sheets.
rename_map = {
    "Invoice": "InvoiceNo",
    "Price": "UnitPrice",
    "Customer ID": "CustomerID"
}
df = df.rename(columns={k:v for k,v in rename_map.items() if k in df.columns})

df["InvoiceNo"] = df["InvoiceNo"].astype(str)
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["CustomerID"] = pd.to_numeric(df["CustomerID"], errors="coerce")

clean = df.loc[
    ~df["InvoiceNo"].str.upper().str.startswith("C")
    & df["InvoiceDate"].notna()
    & df["CustomerID"].notna()
    & (df["Quantity"] > 0)
    & (df["UnitPrice"] > 0)
    & (df["Country"].astype(str).str.strip() == "United Kingdom")
].copy()

clean["CustomerID"] = clean["CustomerID"].astype(int)
clean["revenue"] = clean["Quantity"] * clean["UnitPrice"]

print(f"Clean purchase rows: {len(clean):,}")
print(f"Unique customers: {clean['CustomerID'].nunique():,}")
print(f"Unique invoices: {clean['InvoiceNo'].nunique():,}")
print(f"Observed revenue: £{clean['revenue'].sum():,.2f}")
print(f"Date range: {clean['InvoiceDate'].min()} -> {clean['InvoiceDate'].max()}")

In [ ]:
daily_revenue = clean.set_index("InvoiceDate")["revenue"].resample("W").sum()

fig, ax = plt.subplots(figsize=(12,4.6))
ax.plot(daily_revenue.index, daily_revenue.values)
ax.set_title("Weekly UK online-retail revenue")
ax.set_ylabel("Revenue (£)")
ax.set_xlabel("Week")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

customer_spend = clean.groupby("CustomerID")["revenue"].sum().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8,4.6))
ax.hist(np.log1p(customer_spend.values), bins=50)
ax.set_title("Customer lifetime spend distribution")
ax.set_xlabel("log(1 + observed spend)")
ax.set_ylabel("Customers")
plt.tight_layout()
plt.show()

## 3. Build leakage-safe customer snapshots

Each snapshot uses the previous **180 days** of behaviour and labels what happens during the **next 90 days**. Multiple historical cutoffs create a proper supervised learning table rather than training on one arbitrary end-of-file snapshot.

In [ ]:
LOOKBACK_DAYS = 180
HORIZON_DAYS = 90

cutoffs = pd.to_datetime([
    "2010-09-01",
    "2010-12-01",
    "2011-03-01",
    "2011-06-01",
    "2011-09-01"
])

def customer_snapshot(transactions, cutoff):
    hist_start = cutoff - pd.Timedelta(days=LOOKBACK_DAYS)
    future_end = cutoff + pd.Timedelta(days=HORIZON_DAYS)

    hist = transactions.loc[
        (transactions["InvoiceDate"] >= hist_start)
        & (transactions["InvoiceDate"] < cutoff)
    ].copy()

    fut = transactions.loc[
        (transactions["InvoiceDate"] >= cutoff)
        & (transactions["InvoiceDate"] < future_end)
    ].copy()

    if hist.empty:
        return pd.DataFrame()

    # Invoice-level behaviour.
    invoice_level = (
        hist.groupby(["CustomerID","InvoiceNo"])
        .agg(
            invoice_date=("InvoiceDate","min"),
            invoice_revenue=("revenue","sum"),
            invoice_units=("Quantity","sum"),
            invoice_skus=("StockCode","nunique")
        )
        .reset_index()
    )

    cust = (
        hist.groupby("CustomerID")
        .agg(
            last_purchase=("InvoiceDate","max"),
            first_purchase=("InvoiceDate","min"),
            monetary_180d=("revenue","sum"),
            units_180d=("Quantity","sum"),
            product_diversity=("StockCode","nunique"),
            line_items=("StockCode","size")
        )
    )

    inv = (
        invoice_level.groupby("CustomerID")
        .agg(
            frequency_180d=("InvoiceNo","nunique"),
            avg_order_value=("invoice_revenue","mean"),
            max_order_value=("invoice_revenue","max"),
            avg_units_per_order=("invoice_units","mean"),
            avg_skus_per_order=("invoice_skus","mean")
        )
    )

    cust = cust.join(inv)
    cust["recency_days"] = (cutoff - cust["last_purchase"]).dt.days
    cust["customer_age_days"] = (cutoff - cust["first_purchase"]).dt.days.clip(lower=1)
    cust["orders_per_30d"] = cust["frequency_180d"] / (cust["customer_age_days"]/30).clip(lower=1)
    cust["spend_per_order"] = cust["monetary_180d"] / cust["frequency_180d"].clip(lower=1)
    cust["units_per_line"] = cust["units_180d"] / cust["line_items"].clip(lower=1)

    # Purchase cadence from invoice dates.
    cadence = {}
    for cid, g in invoice_level.groupby("CustomerID"):
        dates = np.sort(g["invoice_date"].dt.normalize().unique())
        if len(dates) >= 2:
            gaps = np.diff(dates).astype("timedelta64[D]").astype(int)
            cadence[cid] = (float(np.mean(gaps)), float(np.std(gaps)))
        else:
            cadence[cid] = (float(LOOKBACK_DAYS), 0.0)

    cadence_df = pd.DataFrame.from_dict(
        cadence, orient="index", columns=["avg_days_between_orders","std_days_between_orders"]
    )
    cust = cust.join(cadence_df)

    future = (
        fut.groupby("CustomerID")
        .agg(
            future_90d_revenue=("revenue","sum"),
            future_90d_orders=("InvoiceNo","nunique")
        )
    )

    cust = cust.join(future, how="left")
    cust[["future_90d_revenue","future_90d_orders"]] = (
        cust[["future_90d_revenue","future_90d_orders"]].fillna(0.0)
    )
    cust["returns_90d"] = (cust["future_90d_orders"] > 0).astype(int)
    cust["cutoff"] = cutoff
    cust = cust.reset_index()

    return cust

snapshots = pd.concat(
    [customer_snapshot(clean, c) for c in cutoffs],
    ignore_index=True
)

print(f"Customer snapshots: {len(snapshots):,}")
print(f"Distinct customers represented: {snapshots['CustomerID'].nunique():,}")
print(f"90-day repeat rate: {snapshots['returns_90d'].mean():.2%}")
print(f"Total labelled future revenue: £{snapshots['future_90d_revenue'].sum():,.2f}")

display(
    snapshots.groupby("cutoff")
    .agg(
        customers=("CustomerID","size"),
        repeat_rate=("returns_90d","mean"),
        future_revenue=("future_90d_revenue","sum")
    )
)

## 4. Time-based train / validation / test split

The latest snapshot is the untouched test cohort. The immediately preceding snapshot is validation. Older cutoffs train the models.

In [ ]:
feature_cols = [
    "monetary_180d","units_180d","product_diversity","line_items",
    "frequency_180d","avg_order_value","max_order_value",
    "avg_units_per_order","avg_skus_per_order","recency_days",
    "customer_age_days","orders_per_30d","spend_per_order",
    "units_per_line","avg_days_between_orders","std_days_between_orders"
]

train_cutoffs = cutoffs[:3]
val_cutoff = cutoffs[3]
test_cutoff = cutoffs[4]

train_df = snapshots.loc[snapshots["cutoff"].isin(train_cutoffs)].copy()
val_df = snapshots.loc[snapshots["cutoff"] == val_cutoff].copy()
test_df = snapshots.loc[snapshots["cutoff"] == test_cutoff].copy()

X_train = train_df[feature_cols].replace([np.inf,-np.inf], np.nan).fillna(0.0)
X_val = val_df[feature_cols].replace([np.inf,-np.inf], np.nan).fillna(0.0)
X_test = test_df[feature_cols].replace([np.inf,-np.inf], np.nan).fillna(0.0)

y_train_return = train_df["returns_90d"].astype(int)
y_val_return = val_df["returns_90d"].astype(int)
y_test_return = test_df["returns_90d"].astype(int)

y_train_value = train_df["future_90d_revenue"].clip(lower=0).astype(float)
y_val_value = val_df["future_90d_revenue"].clip(lower=0).astype(float)
y_test_value = test_df["future_90d_revenue"].clip(lower=0).astype(float)

display(pd.DataFrame({
    "rows":[len(train_df),len(val_df),len(test_df)],
    "repeat_rate":[y_train_return.mean(),y_val_return.mean(),y_test_return.mean()],
    "future_revenue":[y_train_value.sum(),y_val_value.sum(),y_test_value.sum()]
}, index=["train","validation","test"]))

print(f"Model features: {len(feature_cols)}")
print(feature_cols)

## 5. Repeat-purchase propensity: linear baseline vs XGBoost

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

logit = LogisticRegression(
    class_weight="balanced",
    max_iter=2000,
    random_state=RANDOM_STATE
)
logit.fit(X_train_scaled, y_train_return)

xgb_return = XGBClassifier(
    n_estimators=350,
    max_depth=4,
    learning_rate=0.04,
    subsample=0.9,
    colsample_bytree=0.9,
    min_child_weight=3,
    reg_lambda=2.0,
    reg_alpha=0.05,
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=-1
)
xgb_return.fit(X_train, y_train_return)

val_prob_logit = logit.predict_proba(X_val_scaled)[:,1]
val_prob_xgb = xgb_return.predict_proba(X_val)[:,1]

def propensity_metrics(name, y, p, threshold=0.5):
    pred = (np.asarray(p) >= threshold).astype(int)
    return {
        "model":name,
        "ROC_AUC":roc_auc_score(y,p),
        "PR_AUC":average_precision_score(y,p),
        "precision":precision_score(y,pred,zero_division=0),
        "recall":recall_score(y,pred,zero_division=0),
        "F1":f1_score(y,pred,zero_division=0)
    }

val_propensity = pd.DataFrame([
    propensity_metrics("Logistic Regression", y_val_return, val_prob_logit),
    propensity_metrics("XGBoost", y_val_return, val_prob_xgb)
]).set_index("model").sort_values("PR_AUC", ascending=False)

display(val_propensity.style.format({
    "ROC_AUC":"{:.4f}","PR_AUC":"{:.4f}",
    "precision":"{:.2%}","recall":"{:.2%}","F1":"{:.4f}"
}))

best_propensity_name = val_propensity.index[0]
print("Selected propensity model:", best_propensity_name)

## 6. Predict 90-day customer value

A Tweedie XGBoost regressor models the highly skewed, non-negative future revenue distribution including customers who spend zero during the horizon.

In [ ]:
xgb_value = XGBRegressor(
    n_estimators=450,
    max_depth=4,
    learning_rate=0.035,
    subsample=0.9,
    colsample_bytree=0.9,
    min_child_weight=4,
    reg_lambda=3.0,
    reg_alpha=0.1,
    objective="reg:tweedie",
    tweedie_variance_power=1.35,
    eval_metric="mae",
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=-1
)
xgb_value.fit(X_train, y_train_value)

val_pred_value = np.clip(xgb_value.predict(X_val), 0, None)

# Simple behavioural baseline: recent 180-day spend scaled to 90 days.
val_pred_spend_baseline = np.clip(val_df["monetary_180d"].to_numpy() * 0.5, 0, None)

def value_metrics(name, y, pred):
    y = np.asarray(y, dtype=float)
    p = np.asarray(pred, dtype=float)
    return {
        "model":name,
        "MAE":mean_absolute_error(y,p),
        "RMSE":mean_squared_error(y,p)**0.5,
        "WMAPE":np.abs(y-p).sum()/max(np.abs(y).sum(),1e-12),
        "bias":(p-y).sum()/max(y.sum(),1e-12)
    }

val_value_results = pd.DataFrame([
    value_metrics("Historical spend baseline", y_val_value, val_pred_spend_baseline),
    value_metrics("XGBoost Tweedie CLV", y_val_value, val_pred_value)
]).set_index("model").sort_values("MAE")

display(val_value_results.style.format({
    "MAE":"£{:,.2f}","RMSE":"£{:,.2f}",
    "WMAPE":"{:.2%}","bias":"{:+.2%}"
}))

fig, ax = plt.subplots(figsize=(8,4.8))
val_value_results["MAE"].plot(kind="bar", ax=ax)
ax.set_title("Validation 90-day customer-value MAE")
ax.set_ylabel("MAE (£)")
ax.tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.show()

## 7. Untouched test evaluation

In [ ]:
test_prob_logit = logit.predict_proba(X_test_scaled)[:,1]
test_prob_xgb = xgb_return.predict_proba(X_test)[:,1]
test_prob = test_prob_xgb if best_propensity_name == "XGBoost" else test_prob_logit

test_propensity = propensity_metrics(
    best_propensity_name,
    y_test_return,
    test_prob
)

test_pred_value = np.clip(xgb_value.predict(X_test), 0, None)
test_spend_baseline = np.clip(test_df["monetary_180d"].to_numpy() * 0.5, 0, None)

test_value_results = pd.DataFrame([
    value_metrics("Historical spend baseline", y_test_value, test_spend_baseline),
    value_metrics("XGBoost Tweedie CLV", y_test_value, test_pred_value)
]).set_index("model").sort_values("MAE")

print("Repeat-purchase propensity:")
display(pd.DataFrame([test_propensity]).set_index("model").style.format({
    "ROC_AUC":"{:.4f}","PR_AUC":"{:.4f}",
    "precision":"{:.2%}","recall":"{:.2%}","F1":"{:.4f}"
}))

print("90-day value prediction:")
display(test_value_results.style.format({
    "MAE":"£{:,.2f}","RMSE":"£{:,.2f}",
    "WMAPE":"{:.2%}","bias":"{:+.2%}"
}))

## 8. Does predicted CLV rank valuable customers effectively?

For CRM, perfect point estimates are less important than identifying the customers who concentrate future revenue.

In [ ]:
ranking = test_df[["CustomerID","future_90d_revenue"]].copy()
ranking["predicted_90d_value"] = test_pred_value
ranking = ranking.sort_values("predicted_90d_value", ascending=False).reset_index(drop=True)

n = len(ranking)
ranking["decile"] = np.minimum((np.arange(n) * 10 // max(n,1)) + 1, 10)

decile_report = (
    ranking.groupby("decile")
    .agg(
        customers=("CustomerID","size"),
        predicted_value=("predicted_90d_value","sum"),
        actual_future_revenue=("future_90d_revenue","sum")
    )
)
decile_report["actual_revenue_share"] = (
    decile_report["actual_future_revenue"]
    / decile_report["actual_future_revenue"].sum()
)

top10_share = decile_report.loc[1,"actual_revenue_share"]
random_share = 0.10
lift_top_decile = top10_share / random_share

display(decile_report.style.format({
    "predicted_value":"£{:,.0f}",
    "actual_future_revenue":"£{:,.0f}",
    "actual_revenue_share":"{:.2%}"
}))

print(f"Top predicted decile captured {top10_share:.2%} of actual future revenue.")
print(f"Top-decile revenue lift vs random selection: {lift_top_decile:.2f}x")

fig, ax = plt.subplots(figsize=(8,4.8))
ax.bar(decile_report.index.astype(str), decile_report["actual_revenue_share"]*100)
ax.set_title("Actual future revenue captured by predicted CLV decile")
ax.set_xlabel("Predicted value decile (1 = highest)")
ax.set_ylabel("Actual 90-day revenue share (%)")
plt.tight_layout()
plt.show()

## 9. Explainability — behavioural drivers of value and retention

In [ ]:
import shap

value_importance = pd.Series(
    xgb_value.feature_importances_,
    index=feature_cols
).sort_values(ascending=False)

return_importance = pd.Series(
    xgb_return.feature_importances_,
    index=feature_cols
).sort_values(ascending=False)

importance = pd.DataFrame({
    "CLV_importance":value_importance,
    "return_propensity_importance":return_importance
}).fillna(0)

display(importance.head(15))

fig, ax = plt.subplots(figsize=(8,6))
importance["CLV_importance"].head(12).sort_values().plot(kind="barh", ax=ax)
ax.set_title("Top behavioural drivers of predicted customer value")
ax.set_xlabel("XGBoost importance")
plt.tight_layout()
plt.show()

sample = X_val.sample(min(800,len(X_val)),random_state=RANDOM_STATE)
explainer = shap.TreeExplainer(xgb_value)
shap_values = explainer.shap_values(sample)
shap.summary_plot(shap_values, sample, max_display=15, show=False)
plt.title("SHAP — drivers of 90-day customer value")
plt.tight_layout()
plt.show()

## 10. Next-best-action policy

The policy turns **predicted 90-day value + return propensity + recency** into a CRM action.

These are prioritisation rules, not causal uplift estimates. A production system would validate each intervention with randomised experiments.

In [ ]:
actions = test_df[[
    "CustomerID","recency_days","frequency_180d","monetary_180d",
    "product_diversity","future_90d_revenue"
]].copy()

actions["return_probability"] = test_prob
actions["predicted_90d_value"] = test_pred_value

value_hi = actions["predicted_90d_value"].quantile(0.75)
value_mid = actions["predicted_90d_value"].quantile(0.40)
risk_cut = 0.45

def choose_action(row):
    value = row["predicted_90d_value"]
    prob = row["return_probability"]
    recency = row["recency_days"]

    if value >= value_hi and prob < risk_cut:
        return "VIP retention outreach"
    if value >= value_hi and prob >= risk_cut:
        return "Cross-sell / early-access offer"
    if value >= value_mid and prob < risk_cut:
        return "Win-back campaign"
    if prob >= 0.60 and value < value_hi:
        return "Basket expansion recommendation"
    if recency >= 90:
        return "Low-cost reactivation"
    return "Low-cost nurture"

actions["next_best_action"] = actions.apply(choose_action, axis=1)

# Priority emphasises value at risk.
actions["value_at_risk"] = (
    actions["predicted_90d_value"] * (1-actions["return_probability"])
)

def anon_key(cid):
    return "CUST-" + hashlib.sha1(str(int(cid)).encode()).hexdigest()[:8].upper()

actions["customer_key"] = actions["CustomerID"].map(anon_key)

action_summary = (
    actions.groupby("next_best_action")
    .agg(
        customers=("CustomerID","size"),
        avg_predicted_value=("predicted_90d_value","mean"),
        avg_return_probability=("return_probability","mean"),
        total_value_at_risk=("value_at_risk","sum")
    )
    .sort_values("total_value_at_risk", ascending=False)
)

display(action_summary.style.format({
    "avg_predicted_value":"£{:,.2f}",
    "avg_return_probability":"{:.2%}",
    "total_value_at_risk":"£{:,.0f}"
}))

fig, ax = plt.subplots(figsize=(9,5))
plot_actions = action_summary.sort_values("total_value_at_risk")
ax.barh(plot_actions.index, plot_actions["total_value_at_risk"])
ax.set_title("Value-at-risk by recommended CRM action")
ax.set_xlabel("Predicted value at risk (£)")
plt.tight_layout()
plt.show()

## 11. Prioritised CRM action queue

In [ ]:
queue = actions[[
    "customer_key","recency_days","frequency_180d","monetary_180d",
    "product_diversity","return_probability","predicted_90d_value",
    "value_at_risk","next_best_action"
]].sort_values(
    ["value_at_risk","predicted_90d_value"],
    ascending=False
).reset_index(drop=True)

print("Top 30 customer actions:")
display(queue.head(30).style.format({
    "monetary_180d":"£{:,.2f}",
    "return_probability":"{:.2%}",
    "predicted_90d_value":"£{:,.2f}",
    "value_at_risk":"£{:,.2f}"
}))

queue.to_csv("customer_next_best_action_queue.csv", index=False)
action_summary.to_csv("customer_action_summary.csv")
test_value_results.to_csv("customer_clv_test_metrics.csv")
pd.DataFrame([test_propensity]).to_csv("customer_return_propensity_metrics.csv", index=False)

print("CRM decision artefacts exported.")

## 12. Executive summary

In [ ]:
clv_test = test_value_results.loc["XGBoost Tweedie CLV"]
baseline_test = test_value_results.loc["Historical spend baseline"]
top_action = action_summary.index[0]

print(f"""
CUSTOMER LIFETIME VALUE & NEXT-BEST-ACTION SUMMARY
--------------------------------------------------
Dataset: UCI Online Retail II
Modelling approach: behavioural customer snapshots
Lookback window: {LOOKBACK_DAYS} days
Prediction horizon: {HORIZON_DAYS} days
Test customers: {len(test_df):,}

Demographic predictors used: NONE
Raw CustomerID used as a model feature: NO

Selected repeat-propensity model: {best_propensity_name}
Test ROC-AUC: {test_propensity['ROC_AUC']:.4f}
Test PR-AUC: {test_propensity['PR_AUC']:.4f}

CLV test MAE: £{clv_test['MAE']:,.2f}
Historical-spend baseline MAE: £{baseline_test['MAE']:,.2f}
CLV MAE change vs baseline: {(clv_test['MAE']/baseline_test['MAE']-1):+.2%}

Top predicted-value decile captured: {top10_share:.2%} of actual future revenue
Top-decile revenue lift vs random: {lift_top_decile:.2f}x
Largest value-at-risk action pool: {top_action}

The notebook turns customer behaviour into a practical CRM decision system:
transaction history -> behavioural features -> return propensity ->
predicted 90-day value -> value-at-risk -> next-best-action queue.
""")

### Production roadmap
- Gamma-Gamma / BG-NBD benchmark for probabilistic CLV;
- product/category embeddings for personalised cross-sell;
- causal uplift modelling rather than heuristic action assignment;
- contact-cost and gross-margin optimisation;
- channel preference and suppression rules;
- A/B testing by action type;
- calibrated return probabilities;
- drift monitoring and monthly snapshot retraining;
- CRM / marketing-automation integration.

### Interview defence
**“I built customer snapshots at multiple historical cutoffs so every feature precedes the outcome. I excluded demographics and raw customer IDs from modelling, predicted both repeat-purchase propensity and 90-day customer value, evaluated ranking lift as well as point error, and translated the predictions into an anonymised next-best-action queue. I explicitly treat the action policy as prioritisation, not causal uplift, until interventions are experimentally validated.”**